In [0]:
%run ./01-config

In [0]:
import time

# --- Configuration ---

start = int(time.time())

# === CREATE DATABASE ===
print(f"Creating database {catalog}.{db_name}...", end='')
spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.{db_name}")
spark.sql(f"USE {catalog}.{db_name}")
print("Done")

# === BRONZE LAYER (Raw Ingestion Tables) ===

# Bronze: Raw user registration events
print("Creating registered_users_bz table...", end='')
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.registered_users_bz (
        user_id LONG,
        device_id LONG,
        mac_address STRING,
        registration_timestamp DOUBLE,
        load_time TIMESTAMP,
        source_file STRING
    )
""")
print("Done")

# Bronze: Raw gym login/logout events
print("Creating gym_logins_bz table...", end='')
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.{db_name}.gym_logins_bz (
        mac_address STRING,
        gym BIGINT,
        login DOUBLE,
        logout DOUBLE,
        load_time TIMESTAMP,
        source_file STRING
    )
""")
print("Done")

# Bronze: Raw Kafka messages (partitioned for efficient batch reads)
print("Creating kafka_multiplex_bz table...", end='')
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.kafka_multiplex_bz (
        key STRING,
        value STRING,
        topic STRING,
        partition BIGINT,
        offset BIGINT,
        timestamp BIGINT,
        date DATE,
        week_part STRING,
        load_time TIMESTAMP,
        source_file STRING
    )
    PARTITIONED BY (topic, week_part)
""")
print("Done")

# === SILVER LAYER (Cleaned & Conformed Tables) ===

# Silver: Cleaned user registrations
print("Creating users table...", end='')
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.{db_name}.users (
        user_id BIGINT,
        device_id BIGINT,
        mac_address STRING,
        registration_timestamp TIMESTAMP
    )
""")
print("Done")

# Silver: Gym logs with proper timestamps
print("Creating gym_logs table...", end='')
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.{db_name}.gym_logs (
        mac_address STRING,
        gym BIGINT,
        login TIMESTAMP,
        logout TIMESTAMP
    )
""")
print("Done")

# Silver: User demographic/profile data
print("Creating user_profile table...", end='')
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.user_profile (
        user_id BIGINT,
        dob DATE,
        sex STRING,
        gender STRING,
        first_name STRING,
        last_name STRING,
        street_address STRING,
        city STRING,
        state STRING,
        zip INT,
        updated TIMESTAMP
    )
""")
print("Done")

# Silver: Validated heart rate sensor readings
print("Creating heart_rate table...", end='')
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.heart_rate (
        device_id LONG,
        time TIMESTAMP,
        heartrate DOUBLE,
        valid BOOLEAN
    )
""")
print("Done")

# Silver: User demographic bins for segmentation
print("Creating user_bins table...", end='')
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.user_bins (
        user_id BIGINT,
        age STRING,
        gender STRING,
        city STRING,
        state STRING
    )
""")
print("Done")

# Silver: Workout events (start, pause, stop)
print("Creating workouts table...", end='')
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.workouts (
        user_id INT,
        workout_id INT,
        time TIMESTAMP,
        action STRING,
        session_id INT
    )
""")
print("Done")

# Silver: Completed workout sessions
print("Creating completed_workouts table...", end='')
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.completed_workouts (
        user_id INT,
        workout_id INT,
        session_id INT,
        start_time TIMESTAMP,
        end_time TIMESTAMP
    )
""")
print("Done")

# Silver: Heart rate readings linked to workout sessions
print("Creating workout_bpm table...", end='')
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.workout_bpm (
        user_id INT,
        workout_id INT,
        session_id INT,
        start_time TIMESTAMP,
        end_time TIMESTAMP,
        time TIMESTAMP,
        heartrate DOUBLE
    )
""")
print("Done")

# Silver: Date dimension lookup table
print("Creating date_lookup table...", end='')
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.date_lookup (
        date DATE,
        week INT,
        year INT,
        month INT,
        dayofweek INT,
        dayofmonth INT,
        dayofyear INT,
        week_part STRING
    )
""")
print("Done")

# === GOLD LAYER (Business Aggregations & Views) ===

# Gold: Workout BPM summary with demographic context
print("Creating workout_bpm_summary table...", end='')
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.workout_bpm_summary (
        workout_id INT,
        session_id INT,
        user_id BIGINT,
        age STRING,
        gender STRING,
        city STRING,
        state STRING,
        min_bpm DOUBLE,
        avg_bpm DOUBLE,
        max_bpm DOUBLE,
        num_recordings BIGINT
    )
""")
print("Done")

# Gold: Gym usage summary view
print("Creating gym_summary view...", end='')
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.{db_name}.gym_summary AS
    SELECT
        to_date(login::timestamp) AS date,
        gym,
        l.mac_address,
        workout_id,
        session_id,
        round((logout::long - login::long) / 60, 2) AS minutes_in_gym,
        round((end_time::long - start_time::long) / 60, 2) AS minutes_exercising
    FROM {catalog}.{db_name}.gym_logs l
    JOIN (
        SELECT mac_address, workout_id, session_id, start_time, end_time
        FROM {catalog}.{db_name}.completed_workouts w
        INNER JOIN {catalog}.{db_name}.users u ON w.user_id = u.user_id
    ) w
    ON l.mac_address = w.mac_address
    AND w.start_time BETWEEN l.login AND l.logout
    ORDER BY date, gym, l.mac_address, session_id
""")
print("Done")

print(f"\nSetup completed in {int(time.time()) - start} seconds")

In [0]:
start = int(time.time())
# Assert that a specific non-temporary table exists in the target database (batch, no function)
exists = spark.sql(f"SHOW TABLES IN {catalog}.{db_name}") \
             .filter(f"isTemporary == false AND tableName == 'registered_users_bz'") \
             .count() == 1
assert exists, f"The table registered_users_bz is missing in {catalog}.{db_name}"
print(f"Found registered_users_bz table in {catalog}.{db_name}: Success")

exists = spark.sql(f"SHOW TABLES IN {catalog}.{db_name}") \
             .filter(f"isTemporary == false AND tableName == 'gym_logins_bz'") \
             .count() == 1
assert exists, f"The table gym_logins_bz is missing in {catalog}.{db_name}"
print(f"Found gym_logins_bz table in {catalog}.{db_name}: Success")

exists = spark.sql(f"SHOW TABLES IN {catalog}.{db_name}") \
             .filter(f"isTemporary == false AND tableName == 'kafka_multiplex_bz'") \
             .count() == 1
assert exists, f"The table kafka_multiplex_bz is missing in {catalog}.{db_name}"
print(f"Found kafka_multiplex_bz table in {catalog}.{db_name}: Success")

exists = spark.sql(f"SHOW TABLES IN {catalog}.{db_name}") \
             .filter(f"isTemporary == false AND tableName == 'users'") \
             .count() == 1
assert exists, f"The table users is missing in {catalog}.{db_name}"
print(f"Found users table in {catalog}.{db_name}: Success")

exists = spark.sql(f"SHOW TABLES IN {catalog}.{db_name}") \
             .filter(f"isTemporary == false AND tableName == 'gym_logs'") \
             .count() == 1
assert exists, f"The table gym_logs is missing in {catalog}.{db_name}"
print(f"Found gym_logs table in {catalog}.{db_name}: Success")

exists = spark.sql(f"SHOW TABLES IN {catalog}.{db_name}") \
             .filter(f"isTemporary == false AND tableName == 'user_profile'") \
             .count() == 1
assert exists, f"The table user_profile is missing in {catalog}.{db_name}"
print(f"Found user_profile table in {catalog}.{db_name}: Success")

exists = spark.sql(f"SHOW TABLES IN {catalog}.{db_name}") \
             .filter(f"isTemporary == false AND tableName == 'heart_rate'") \
             .count() == 1
assert exists, f"The table heart_rate is missing in {catalog}.{db_name}"
print(f"Found heart_rate table in {catalog}.{db_name}: Success")

exists = spark.sql(f"SHOW TABLES IN {catalog}.{db_name}") \
             .filter(f"isTemporary == false AND tableName == 'workouts'") \
             .count() == 1
assert exists, f"The table workouts is missing in {catalog}.{db_name}"
print(f"Found workouts table in {catalog}.{db_name}: Success")

exists = spark.sql(f"SHOW TABLES IN {catalog}.{db_name}") \
             .filter(f"isTemporary == false AND tableName == 'completed_workouts'") \
             .count() == 1
assert exists, f"The table completed_workouts is missing in {catalog}.{db_name}"
print(f"Found completed_workouts table in {catalog}.{db_name}: Success")

exists = spark.sql(f"SHOW TABLES IN {catalog}.{db_name}") \
             .filter(f"isTemporary == false AND tableName == 'workout_bpm'") \
             .count() == 1
assert exists, f"The table workout_bpm is missing in {catalog}.{db_name}"
print(f"Found workout_bpm table in {catalog}.{db_name}: Success")

exists = spark.sql(f"SHOW TABLES IN {catalog}.{db_name}") \
             .filter(f"isTemporary == false AND tableName == 'user_bins'") \
             .count() == 1
assert exists, f"The table user_bins is missing in {catalog}.{db_name}"
print(f"Found user_bins table in {catalog}.{db_name}: Success")

exists = spark.sql(f"SHOW TABLES IN {catalog}.{db_name}") \
             .filter(f"isTemporary == false AND tableName == 'date_lookup'") \
             .count() == 1
assert exists, f"The table date_lookup is missing in {catalog}.{db_name}"
print(f"Found date_lookup table in {catalog}.{db_name}: Success")

exists = spark.sql(f"SHOW TABLES IN {catalog}.{db_name}") \
             .filter(f"isTemporary == false AND tableName == 'workout_bpm_summary'") \
             .count() == 1
assert exists, f"The table workout_bpm_summary is missing in {catalog}.{db_name}"
print(f"Found workout_bpm_summary table in {catalog}.{db_name}: Success")

exists = spark.sql(f"SHOW TABLES IN {catalog}.{db_name}") \
             .filter(f"isTemporary == false AND tableName == 'gym_summary'") \
             .count() == 1
assert exists, f"The table gym_summary is missing in {catalog}.{db_name}"
print(f"Found gym_summary table in {catalog}.{db_name}: Success")

# Validate database exists (batch, no function)
db_exists = spark.sql(f"SHOW DATABASES IN {catalog}") \
                 .filter(f"databaseName == '{db_name}'") \
                 .count() == 1
assert db_exists, f"The database '{catalog}.{db_name}' is missing"
print(f"Found database {catalog}.{db_name}: Success")

print(f"\nSetup completed in {int(time.time()) - start} seconds")



In [0]:
# Cleanup: drop database and all tables/views (batch, no function)
#try:
    #if spark.sql(f"SHOW DATABASES IN {catalog}") \
     #       .filter(f"databaseName == '{db_name}'").count() == 1:
     #   print(f"Dropping the database {catalog}.{db_name}...", end='')
     #   spark.catalog.clearCache()
     #   spark.sql(f"DROP DATABASE {catalog}.{db_name} CASCADE")
     #   print("Done")
    #else:
    #    print(f"Database {catalog}.{db_name} does not exist. Skipping drop.")
#except Exception as e:
 #   print(f"Error while dropping database: {str(e)}")